In [13]:
# ==============================
# INSTALL DEPENDENCIES
# ==============================
!pip install requests beautifulsoup4 reportlab

# ==============================
# IMPORTS
# ==============================
import requests
from bs4 import BeautifulSoup
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
import os
from urllib.parse import urljoin

# ==============================
# INPUT URLS
# ==============================
urls = [
    "https://www.sastra.edu/student-activities/kuruksastra.html",
    "https://www.sastra.edu/student-activities/daksh.html",
    "https://www.sastra.edu/student-activities/clubs.html",
    "https://sastra.edu/orgagen/",
    "https://www.sastra.edu/parents-corner/library-orientation-programme.html",
    "https://www.sastra.edu/hrpolicy/",
    "https://www.sastra.edu/gh/",
    "https://www.sastra.edu/useful-links/anti-drug-club.html",
    "https://www.sastra.edu/admissions/ug-pg.html",
    "https://www.sastra.edu/admissions/eligibility-criteria.html",
    "https://www.sastra.edu/admissions/fee-structure.html",
    "https://www.sastra.edu/admissions/hostel-fees.html",
    "https://www.sastra.edu/admissions.html",
    "https://www.sastra.edu/admissions/admissions-faq.html",
    "https://www.sastra.edu/academics/schools.html",
    "https://www.sastra.edu/academics/sastra-ai.html"
]

OUTPUT_DIR = "pdf_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

visited = set()
styles = getSampleStyleSheet()

# ==============================
# FUNCTION: SAVE TEXT TO PDF
# ==============================
def save_pdf(text, filename):
    doc = SimpleDocTemplate(filename)
    story = []

    for line in text.split("\n"):
        if line.strip():
            story.append(Paragraph(line, styles["Normal"]))
            story.append(Spacer(1, 10))

    doc.build(story)

# ==============================
# FUNCTION: SCRAPE PAGE
# ==============================
def scrape(url):
    try:
        print(f"🔍 Scraping: {url}")
        response = requests.get(url, timeout=20)

        if response.status_code != 200:
            print("❌ Failed request")
            return []

        soup = BeautifulSoup(response.text, "html.parser")

        # Extract text
        text = soup.get_text(separator="\n")

        if len(text.strip()) < 200:
            print("⚠️ Skipped (too little content)")
            return []

        # Save PDF
        file_name = url.replace("https://", "").replace("/", "_")[:100] + ".pdf"
        path = os.path.join(OUTPUT_DIR, file_name)

        save_pdf(text, path)
        print(f"✅ Saved: {path}")

        # Extract sublinks
        sublinks = []
        for a in soup.find_all("a", href=True):
            link = urljoin(url, a['href'])

            if "sastra.edu" in link:
                sublinks.append(link)

        return sublinks

    except Exception as e:
        print(f"❌ Error: {e}")
        return []

# ==============================
# MAIN CRAWLER
# ==============================
queue = urls.copy()

MAX_PAGES = 30  # safety limit

while queue and len(visited) < MAX_PAGES:
    current = queue.pop(0)

    if current in visited:
        continue

    visited.add(current)

    sublinks = scrape(current)

    for link in sublinks:
        if link not in visited:
            queue.append(link)

# ==============================
# CHECK FILES
# ==============================
print("\n📁 Files created:")
print(os.listdir(OUTPUT_DIR))

# ==============================
# ZIP + DOWNLOAD
# ==============================
import shutil
from google.colab import files

shutil.make_archive("sastra_pdfs", 'zip', OUTPUT_DIR)
files.download("sastra_pdfs.zip")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 51.8 MB/s eta 0:00:00
🔍 Scraping: https://www.sastra.edu/student-activities/kuruksastra.html
✅ Saved: pdf_outputs/www.sastra.edu_student-activities_kuruksastra.html.pdf
🔍 Scraping: https://www.sastra.edu/student-activities/daksh.html
✅ Saved: pdf_outputs/www.sastra.edu_student-activities_daksh.html.pdf
🔍 Scraping: https://www.sastra.edu/student-activities/clubs.html
✅ Saved: pdf_outputs/www.sastra.edu_student-activities_clubs.html.pdf
🔍 Scraping: https://sastra.edu/orgagen/
✅ Saved: pdf_outputs/sastra.edu_orgagen_.pdf
🔍 Scraping: https://www.sastra.edu/parents-corner/library-orientation-programme.html
✅ Saved: pdf_outputs/www.sastra.edu_parents-corner_library-orientation-programme.html.pdf
🔍 Scraping: https://www.sastra.edu/hrpolicy/
✅ Saved: pdf_outputs/www.sastra.edu_hrpolicy_.pdf
🔍 Scraping: https://www.sastra.edu/gh/
✅ Saved: pdf_outputs/www.sastra.edu_gh_.pdf
🔍 Scraping: https://www.sastra.edu/useful-links/anti-drug-club.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>